# Nemotron 3.5 Lightning 30B

Calls `nvidia/nemotron-3.5-lightning-30b-a3b` on NVIDIA NIM. Lightning is the fast model in this set: 30B total parameters, about 3B active, with a hybrid Mamba-2 and Mixture-of-Experts architecture. Use it when you want a reasoning reply without waiting on the 550B model.

## Runtime

This notebook runs on any Colab runtime, including the Colab Pro **A100 GPU** runtime. Pick it with **Runtime → Change runtime type → A100 GPU**. The first code cell reports the attached GPU.

The model itself is a hosted NVIDIA NIM. Colab sends the prompt to `https://integrate.api.nvidia.com` and NVIDIA's data-center GPUs do the inference, so the notebook finishes in seconds on an A100 runtime and the same code also runs on a CPU runtime. If Colab reports that no A100 is available, the fallback is to run on CPU. The reply will be identical because the model does not run on the Colab VM.

## API key

The notebook reads a Colab secret named `NVIDIA_API_KEY`.

1. In Colab, click the key icon in the left sidebar.
2. Add a secret named `NVIDIA_API_KEY`.
3. Paste a key from [build.nvidia.com](https://build.nvidia.com/).
4. Enable notebook access for that secret.
5. Choose **Runtime → Change runtime type → A100 GPU** (or leave CPU).
6. Choose **Runtime → Run all**.

If the secret is not set, the next cell asks for the key with a hidden prompt. The key stays in the runtime only. Do not paste it into a cell you save or commit.


In [1]:
import platform
import shutil
import subprocess

print("Python", platform.python_version())
if shutil.which("nvidia-smi"):
    listed = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    gpu_line = (listed.stdout or listed.stderr).strip() or "nvidia-smi returned no GPU list."
    print("Colab GPU:", gpu_line)
    if "A100" in gpu_line:
        print("A100 runtime detected.")
else:
    print("Colab GPU: none attached (CPU runtime).")
print("Inference runs on NVIDIA NIM data-center GPUs. This VM only sends the request and streams the reply.")


Python 3.13.5
Colab GPU: none attached (CPU runtime).
Inference runs on NVIDIA NIM data-center GPUs. This VM only sends the request and streams the reply.


In [2]:
%pip install -q "openai>=1.54.0,<3" "requests>=2.31.0"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import time
from getpass import getpass

from openai import APIConnectionError, APIStatusError, APITimeoutError, OpenAI


def load_nvidia_api_key() -> str:
    key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            secret = userdata.get("NVIDIA_API_KEY")
        except Exception:
            secret = None
        if secret and str(secret).strip():
            key = str(secret).strip()
            os.environ["NVIDIA_API_KEY"] = key
            return key
    key = getpass("Enter your NVIDIA API key: ").strip()
    if not key:
        raise RuntimeError(
            "No NVIDIA API key was provided. Add a Colab secret named "
            "NVIDIA_API_KEY, allow notebook access, and run this cell again."
        )
    os.environ["NVIDIA_API_KEY"] = key
    return key


def _friendly_api_error(exc: Exception) -> str:
    status = getattr(exc, "status_code", None)
    if status in (401, 403):
        return (
            "NVIDIA rejected the API key. Create a key at https://build.nvidia.com/ "
            "and save it as the Colab secret NVIDIA_API_KEY."
        )
    body = getattr(exc, "body", None)
    return f"NVIDIA NIM request failed ({status or type(exc).__name__}): {body or exc}"


def _delta_field(delta, name: str):
    value = getattr(delta, name, None)
    if value:
        return value
    extra = getattr(delta, "model_extra", None) or {}
    return extra.get(name)


def stream_nim(client: OpenAI, model: str, prompt: str, max_tokens: int, reasoning_budget: int,
               temperature: float = 1, top_p: float = 0.95) -> str:
    """Stream a Nemotron reply. max_tokens must leave room after the reasoning trace."""
    if max_tokens < 1:
        raise ValueError("max_tokens must be at least 1")
    if reasoning_budget < -1:
        raise ValueError("reasoning_budget must be -1 or greater")

    def once() -> str:
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            extra_body={
                "chat_template_kwargs": {"enable_thinking": True},
                "reasoning_budget": reasoning_budget,
            },
            stream=True,
            timeout=300,
        )
        reasoning_parts = []
        content_parts = []
        printed_reasoning = False
        printed_answer = False
        print(f"Model: {model}", flush=True)
        for chunk in completion:
            if not getattr(chunk, "choices", None):
                continue
            delta = chunk.choices[0].delta
            reasoning = _delta_field(delta, "reasoning_content")
            content = _delta_field(delta, "content")
            if reasoning:
                if not printed_reasoning:
                    print("\n--- reasoning ---\n", flush=True)
                    printed_reasoning = True
                reasoning_parts.append(reasoning)
                print(reasoning, end="", flush=True)
            if content:
                if not printed_answer:
                    print("\n\n--- answer ---\n", flush=True)
                    printed_answer = True
                content_parts.append(content)
                print(content, end="", flush=True)
        print("\n", flush=True)
        answer = "".join(content_parts).strip()
        if not answer:
            raise RuntimeError(
                "NVIDIA NIM returned no answer text. Increase max_tokens so it is "
                "larger than the reasoning trace, then run the cell again."
            )
        return answer

    last_error = None
    for attempt in range(2):
        try:
            return once()
        except (APIStatusError, APIConnectionError, APITimeoutError) as exc:
            last_error = exc
            status = getattr(exc, "status_code", None)
            retryable = status in (None, 429, 500, 502, 503, 504)
            if attempt == 1 or not retryable:
                raise RuntimeError(_friendly_api_error(exc)) from None
            print(f"Temporary NIM error ({status or type(exc).__name__}). Retrying once...", flush=True)
            time.sleep(3)
    raise RuntimeError(_friendly_api_error(last_error)) from None


NVIDIA_API_KEY = load_nvidia_api_key()
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=300,
    max_retries=0,
)
print(f"NVIDIA API key is ready ({len(NVIDIA_API_KEY)} characters). The key is not printed.")


NVIDIA API key is ready (70 characters). The key is not printed.


## Ask Lightning

Edit `PROMPT` and run the cell. Reasoning tokens print first, then the answer. Temperature `1` and `top_p` `0.95` are the sampling values published for this model.

In [4]:
# Colab-safe defaults. The API allows up to 32768 tokens.
# Keep MAX_TOKENS larger than the reasoning trace, or the answer comes back empty.
PROMPT = "Write a limerick about the wonders of GPU computing."
MAX_TOKENS = 2048
REASONING_BUDGET = 1024
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"

answer = stream_nim(
    client,
    model=MODEL,
    prompt=PROMPT,
    max_tokens=MAX_TOKENS,
    reasoning_budget=REASONING_BUDGET,
    temperature=1,
    top_p=0.95,
)
print(f"Answer length: {len(answer)} characters")


Model: nvidia/nemotron-3.5-lightning-30b-a3b



--- reasoning ---



Here

's

 a

 thinking

... (936 outputs left out)

 great

 need

.

Answer length: 205 characters
